# 09f — Soluzioni: Function Calling e MCP

**Corso**: Programmazione di Applicazioni Intelligenti  
**Lezione 09** — Dal Client OpenAI al Function Calling e MCP  
**Blocco 4** — Soluzioni dell'esercitazione

Questo notebook contiene le soluzioni commentate degli esercizi del notebook `09e`.


---
## Setup


In [ ]:
!pip install -q openai

from openai import OpenAI
from google.colab import userdata
import json

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
)
MODEL = "openai/gpt-oss-120b"
print(f"Client pronto! Modello: {MODEL}")


---
## Soluzione Esercizio 1 — Function calling base


In [ ]:
# Soluzione 1 — Definizione del tool

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Restituisce il tasso di cambio corrente tra due valute. Usa i codici ISO 4217 (es. EUR, USD, GBP).",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_currency": {
                        "type": "string",
                        "description": "Codice ISO della valuta di partenza (es. EUR)"
                    },
                    "to_currency": {
                        "type": "string",
                        "description": "Codice ISO della valuta di destinazione (es. USD)"
                    }
                },
                "required": ["from_currency", "to_currency"]
            }
        }
    }
]

print("Tool definito:")
print(json.dumps(tools, indent=2, ensure_ascii=False))


In [ ]:
# Soluzione 1 — Funzione Python e registry

def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """Restituisce il tasso di cambio (dati simulati)."""
    rates = {
        "EUR_USD": 1.08,
        "USD_EUR": 0.93,
        "EUR_GBP": 0.86,
        "GBP_EUR": 1.16,
        "USD_GBP": 0.79,
        "GBP_USD": 1.27,
        "EUR_JPY": 162.50,
        "JPY_EUR": 0.0062,
    }

    key = f"{from_currency.upper()}_{to_currency.upper()}"
    if key in rates:
        return json.dumps({
            "from": from_currency.upper(),
            "to": to_currency.upper(),
            "rate": rates[key]
        })
    else:
        return json.dumps({"error": f"Tasso non disponibile per {key}"})


# Registry: mappa nome_tool -> funzione Python
tool_registry = {
    "get_exchange_rate": get_exchange_rate
}

# Verifica
print(get_exchange_rate("EUR", "USD"))
print(get_exchange_rate("EUR", "CHF"))  # non disponibile


In [ ]:
# Soluzione 1 — Loop completo

def ask_with_tools(question: str) -> str:
    """Gestisce una domanda con function calling (singolo round)."""
    messages = [
        {"role": "system", "content": "Sei un assistente finanziario. Usa i tool disponibili. Rispondi in italiano."},
        {"role": "user", "content": question}
    ]

    # Prima chiamata: il modello decide se usare un tool
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    # Se il modello non chiede tool, restituisci direttamente
    if not message.tool_calls:
        return message.content

    # Aggiungi il messaggio dell'assistente (con le tool_calls) alla cronologia
    messages.append(message)

    # Per ogni tool_call, esegui la funzione e aggiungi il risultato
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        print(f"  [Tool call] {fn_name}({fn_args})")

        # Esegui la funzione dal registry
        result = tool_registry[fn_name](**fn_args)
        print(f"  [Risultato] {result}")

        # Aggiungi il risultato con role "tool"
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

    # Seconda chiamata: il modello formula la risposta usando i risultati
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    return final_response.choices[0].message.content

# Test
print(ask_with_tools("Quanto vale 100 euro in dollari?"))


---
## Soluzione Esercizio 2 — Multi-tool


In [ ]:
# Soluzione 2 — Aggiungi il tool translate_text

tools.append({
    "type": "function",
    "function": {
        "name": "translate_text",
        "description": "Traduce un testo nella lingua specificata. Restituisce il testo tradotto.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "Il testo da tradurre"
                },
                "target_language": {
                    "type": "string",
                    "description": "La lingua di destinazione (es. 'inglese', 'francese', 'tedesco')"
                }
            },
            "required": ["text", "target_language"]
        }
    }
})


def translate_text(text: str, target_language: str) -> str:
    """Simulazione di traduzione con un dizionario semplice."""
    # Dizionario di traduzioni note (simulato)
    translations = {
        ("buongiorno", "inglese"): "good morning",
        ("buongiorno", "francese"): "bonjour",
        ("grazie", "inglese"): "thank you",
        ("grazie", "francese"): "merci",
        ("ciao", "inglese"): "hello",
    }

    key = (text.lower().strip(), target_language.lower().strip())
    if key in translations:
        translated = translations[key]
    else:
        # Fallback: prefisso con la lingua (simulazione)
        translated = f"[{target_language}] {text}"

    return json.dumps({
        "original": text,
        "translated": translated,
        "language": target_language
    })


# Aggiungi al registry
tool_registry["translate_text"] = translate_text

print(f"Tool registrati: {list(tool_registry.keys())}")


In [ ]:
# Soluzione 2 — Test con domande diverse

# Test A: solo cambio valuta
print("=== Test A: solo cambio ===")
print(ask_with_tools("Qual e' il tasso di cambio euro-sterlina?"))
print()

# Test B: solo traduzione
print("=== Test B: solo traduzione ===")
print(ask_with_tools("Traduci 'buongiorno' in inglese"))
print()

# Test C: domanda che potrebbe richiedere entrambi
# NOTA: il nostro loop fa un singolo round. Se il modello emette entrambe
# le tool call in parallelo, translate_text non avra' il risultato del cambio.
# Se il modello e' abbastanza smart, chiamara' solo get_exchange_rate nel primo
# round e poi tradurra' da solo nella risposta finale.
print("=== Test C: entrambi? ===")
print(ask_with_tools("Quanto vale 1 euro in dollari? Traduci la risposta in inglese."))


---
## Soluzione Esercizio 3 — MCP Server


In [ ]:
!pip install -q "mcp[cli]"


In [ ]:
# Soluzione 3 — MCP Server con FastMCP

from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("Todo Server")

# Stato in-memory
tasks = []
next_id = 1

@mcp_server.tool()
def add_task(title: str, priority: str = "medium") -> str:
    """Aggiunge un task alla lista."""
    global next_id
    task = {
        "id": next_id,
        "title": title,
        "priority": priority,
        "completed": False
    }
    tasks.append(task)
    next_id += 1
    return f"Task aggiunto: #{task['id']} '{title}' (priorita': {priority})"


@mcp_server.tool()
def list_tasks() -> str:
    """Elenca tutti i task con id, titolo, priorita' e stato."""
    if not tasks:
        return "Nessun task presente."

    lines = []
    for t in tasks:
        stato = "completato" if t["completed"] else "da fare"
        lines.append(f"#{t['id']} [{t['priority']}] {t['title']} — {stato}")
    return "\n".join(lines)


@mcp_server.tool()
def complete_task(task_id: int) -> str:
    """Segna un task come completato dato il suo id."""
    for t in tasks:
        if t["id"] == task_id:
            t["completed"] = True
            return f"Task #{task_id} '{t['title']}' segnato come completato."
    return f"Errore: nessun task con id #{task_id}"


print("Server MCP definito!")


In [ ]:
# Soluzione 3 — Test del server in Colab

import asyncio

async def test_todo_server():
    # 1. Verifica i tool registrati
    print("=== Tool registrati ===")
    tools_list = await mcp_server.list_tools()
    for t in tools_list:
        print(f"  {t.name}: {t.description}")
        print(f"    Schema: {t.inputSchema}")
    print()

    # 2. Aggiungi qualche task
    print("=== Aggiunta task ===")
    r1 = await mcp_server.call_tool("add_task", {"title": "Studiare function calling", "priority": "high"})
    print(f"  {r1[0][0].text}")
    r2 = await mcp_server.call_tool("add_task", {"title": "Fare la spesa"})
    print(f"  {r2[0][0].text}")
    r3 = await mcp_server.call_tool("add_task", {"title": "Leggere paper su ReAct", "priority": "low"})
    print(f"  {r3[0][0].text}")
    print()

    # 3. Lista task
    print("=== Lista task ===")
    r4 = await mcp_server.call_tool("list_tasks", {})
    print(f"  {r4[0][0].text}")
    print()

    # 4. Completa un task
    print("=== Completamento ===")
    r5 = await mcp_server.call_tool("complete_task", {"task_id": 1})
    print(f"  {r5[0][0].text}")
    print()

    # 5. Prova con id inesistente
    print("=== Errore: id inesistente ===")
    r6 = await mcp_server.call_tool("complete_task", {"task_id": 99})
    print(f"  {r6[0][0].text}")
    print()

    # 6. Lista aggiornata
    print("=== Lista aggiornata ===")
    r7 = await mcp_server.call_tool("list_tasks", {})
    print(f"  {r7[0][0].text}")

await test_todo_server()


### Test in locale con MCP Inspector

Per testare il server in locale con l'Inspector, salva il codice in un file `todo_server.py`
(aggiungendo `mcp_server.run()` alla fine) e lancia:

```bash
pip install "mcp[cli]"
npx @modelcontextprotocol/inspector python todo_server.py
```

L'Inspector avvia il server via stdio e ti permette di esplorare i tool visivamente nel browser.


---
## Soluzione Esercizio 4 (Bonus) — Guardrails di sicurezza


In [ ]:
# Soluzione 4 — Versione sicura di calculate

import re

def safe_calculate(expression: str) -> str:
    """
    Calcola un'espressione aritmetica in modo sicuro.
    Accetta solo: cifre, spazi, operatori (+, -, *, /, **, %), parentesi, punto decimale.
    """
    # Regex: accetta solo caratteri sicuri per espressioni aritmetiche
    # - \d: cifre
    # - \s: spazi
    # - \+\-\*/: operatori base
    # - \(\): parentesi
    # - \.: punto decimale
    # - %: modulo
    # Nota: ** (potenza) e' composto da due *, gia' coperti da \*
    allowed_pattern = r'^[\d\s\+\-\*/\(\)\.%]+$'

    if not re.match(allowed_pattern, expression):
        return json.dumps({"error": f"Espressione non valida: {expression}"})

    # Ulteriore controllo: rifiuta espressioni vuote o con solo operatori
    if not re.search(r'\d', expression):
        return json.dumps({"error": f"Espressione senza numeri: {expression}"})

    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": result})
    except ZeroDivisionError:
        return json.dumps({"error": "Divisione per zero"})
    except Exception as e:
        return json.dumps({"error": str(e)})


# Test con input legittimi
print("=== Input legittimi ===")
print(safe_calculate("15 * 23 + 7"))        # {"expression": "15 * 23 + 7", "result": 352}
print(safe_calculate("(100 + 50) / 3"))     # {"expression": "(100 + 50) / 3", "result": 50.0}
print(safe_calculate("2 ** 10"))             # {"expression": "2 ** 10", "result": 1024}
print(safe_calculate("10 % 3"))              # {"expression": "10 % 3", "result": 1}
print(safe_calculate("3.14 * 2"))            # {"expression": "3.14 * 2", "result": 6.28}

# Test con input malevoli
print("\n=== Input malevoli (devono essere rifiutati) ===")
print(safe_calculate("__import__('os').system('rm -rf /')"))  # contiene lettere e apici
print(safe_calculate("eval('print(1)')"))                     # contiene lettere e apici
print(safe_calculate("open('/etc/passwd').read()"))            # contiene lettere e apici

# Test edge case
print("\n=== Edge case ===")
print(safe_calculate("1 / 0"))              # divisione per zero
print(safe_calculate("***"))                 # solo operatori, nessun numero


---
## Riepilogo

### Cosa abbiamo coperto in questa lezione:
- **Blocco 1-2**: la libreria `openai` come interfaccia universale (chat, streaming, reasoning, structured output)
- **Blocco 3-4**: function calling (il meccanismo) e MCP (il protocollo standard)

### Punti chiave dell'esercitazione:
- Il **JSON Schema** descrive i tool al modello: nome, descrizione, parametri tipizzati
- Il **registry** collega i nomi dei tool alle funzioni Python
- Il **loop di function calling** ha una struttura fissa: chiamata → check tool_calls → esecuzione → risposta
- Il loop singolo ha un **limite**: non gestisce dipendenze sequenziali tra tool
- **MCP Server** con FastMCP: `@mcp.tool()` genera automaticamente lo schema dal type hint
- La **sicurezza** dei tool (es. `eval()`) richiede validazione esplicita dell'input

### Prossime lezioni:
- **ReAct**: il pattern che fa *ripetere* il loop tool call autonomamente (loop agentico)
- **Skills e Context Engineering**: come gestire contesti lunghi e dare competenze specifiche all'agente
